## Merged Features (Cross-Table)

Now combining cleaned and feature-enriched tables to create master datasets and derive features that require information from multiple sources.

In [1]:
import pandas as pd

In [2]:
orders = pd.read_csv("../data/feature_engineered/orders_fe.csv")
order_items = pd.read_csv("../data/feature_engineered/order_items_fe.csv")
customers = pd.read_csv("../data/feature_engineered/customers_fe.csv")
sellers = pd.read_csv("../data/feature_engineered/sellers_fe.csv")
products = pd.read_csv("../data/feature_engineered/products_fe.csv")
order_payments = pd.read_csv("../data/feature_engineered/order_payments_fe.csv")
order_reviews = pd.read_csv("../data/feature_engineered/order_reviews_fe.csv")
closed_deals = pd.read_csv("../data/feature_engineered/closed_deals_fe.csv")
marketing_leads = pd.read_csv("../data/feature_engineered/marketing_leads_fe.csv")

product_category_name_translations = pd.read_csv(
    "../data/processed/category_translation_clean.csv"
)
geo_locations = pd.read_csv("../data/processed/geolocation_clean.csv")

Before directly performing a merge, I verify the grains and keys of the tables I have, because this is fundamental to a successful merge. This ensures that the feature engineer tables I previously saved are also read correctly.

In [3]:
tables = {
    "orders": orders,
    "order_items": order_items,
    "customers": customers,
    "sellers": sellers,
    "products": products,
    "order_payments": order_payments,
    "order_reviews": order_reviews,
    "closed_deals": closed_deals,
    "marketing_leads": marketing_leads,
    "geo_locations": geo_locations,
    "category_translation": product_category_name_translations
}

for name, df in tables.items():
    print(f"{name}: {df.shape}")

orders: (99441, 20)
order_items: (112650, 16)
customers: (99441, 7)
sellers: (3095, 6)
products: (32951, 12)
order_payments: (103886, 5)
order_reviews: (98673, 10)
closed_deals: (842, 18)
marketing_leads: (8000, 8)
geo_locations: (1000163, 5)
category_translation: (71, 2)


I also check if the keys are truly unique. Apart from that, I don't expect order_items, order_payments, and order_reviews to be unique because they are in different grains.

In [4]:
print("orders order_id unique:", orders["order_id"].is_unique)
print("customers customer_id unique:", customers["customer_id"].is_unique)
print("sellers seller_id unique:", sellers["seller_id"].is_unique)
print("products product_id unique:", products["product_id"].is_unique)
print("marketing_leads mql_id unique:", marketing_leads["mql_id"].is_unique)
print("closed_deals mql_id unique:", closed_deals["mql_id"].is_unique)

orders order_id unique: True
customers customer_id unique: True
sellers seller_id unique: True
products product_id unique: True
marketing_leads mql_id unique: True
closed_deals mql_id unique: True


In [5]:
orders_before = len(orders)
customers_before = len(customers)

print("Orders:", orders_before)
print("Customers:", customers_before)

Orders: 99441
Customers: 99441


In [6]:
print(
    "Orders with customer_id:",
    orders["customer_id"].notna().sum()
)

print(
    "Unique customer_ids in orders:",
    orders["customer_id"].nunique()
)

Orders with customer_id: 99441
Unique customer_ids in orders: 99441


In [7]:
df_master = orders.merge(
    customers,
    on="customer_id",
    how="left",
    validate="many_to_one"
)

In [8]:
print("Before:", len(orders))
print("After:", len(df_master))

Before: 99441
After: 99441


The `order_items` table stores multiple items within an order on different rows. Therefore, having 112,650 rows is normal. Having the same `order_id` appearing in multiple rows is consistent with the table's structure, as an order can contain multiple items.

In [9]:
print(df_master.shape)

(99441, 26)


In [10]:
order_items.columns

Index(['order_id', 'order_item_id', 'product_id', 'seller_id',
       'shipping_limit_date', 'price', 'freight_value', 'item_total_cost',
       'freight_ratio', 'items_in_order', 'order_total_price',
       'order_total_freight', 'order_total_cost', 'sellers_in_order',
       'is_free_shipping', 'average_item_price'],
      dtype='str')

In [11]:
order_items.shape

(112650, 16)

In this case, directly merging the `order_items` table with `df_master` is an incorrect approach. If an order has 3 items, the row in `df_master` will be repeated 3 times. This results in order-level information, such as customer information and order details, being repeated row by row. This corrupts the `grain` of `df_master` and causes the "1 row = 1 order" property to be lost.

In [12]:
order_items["order_id"].nunique()

98666

In [13]:
order_items["order_id"].duplicated().sum()

np.int64(13984)

In [14]:
order_items.groupby("order_id").size().describe()

count    98666.000000
mean         1.141731
std          0.538452
min          1.000000
25%          1.000000
50%          1.000000
75%          1.000000
max         21.000000
dtype: float64